# `JuMP.jl`: A Brief Initial Introduction
Here we present an overview to using `JuMP.jl`. Note that this content takes inspiration from https://jump.dev/JuMP.jl/stable/tutorials/getting_started/getting_started_with_JuMP/.

## Resources
We will not be able to cover all of `JuMP.jl`'s capabilities today. Good references are:
- The tutorials, examples, manuals, and guides in `JuMP.jl`'s documentation: https://jump.dev/JuMP.jl/stable/
- The Julia optimization forum: https://discourse.julialang.org/c/domain/opt/13
- Julia Programming for Operations Research 2/e (not always up-to-date): https://www.softcover.io/read/7b8eb7d0/juliabook2/introduction

## Installation
Let's get started by installing the necessary packages:


In [ ]:
import Pkg
Pkg.add(["JuMP", "HiGHS", "Ipopt", "MathOptInterface", "SpecialFunctions"])

Here `HiGHS` acts an appropriate LP solver and `Ipopt` acts as an appropriate NLP solver. The list of supported solvers and the problems types they can solve is provided at https://jump.dev/JuMP.jl/stable/installation/#Supported-solvers.

## Motivating Example
Consider the following linear program (LP):
$$
\begin{aligned}
& \min && 12x + 20y \\
& \;\;\text{s.t.} && 6x + 8y \geq 100 \\
&&& 7x + 12y \geq 120 \\
&&& x \geq 0 \\
&&& y \in [0, 3] \\
\end{aligned}
$$
Let's formulate this problem in `JuMP.jl` and use the HiGHS solver:

In [ ]:
using JuMP, HiGHS

model = Model(HiGHS.Optimizer)

@variable(model, x >= 0)
@variable(model, 0 <= y <= 3)

@objective(model, Min, 12x + 20y)

@constraint(model, c1, 6x + 8y >= 100)
@constraint(model, c2, 7x + 12y >= 120)

print(model)

latex_formulation(model)

That's all we have to do formulate the model and view it. Now let's optimize it!

In [ ]:
optimize!(model)

@show termination_status(model)
@show primal_status(model)
@show dual_status(model)
@show objective_value(model)
@show value(x)
@show value(y)
@show shadow_price(c1)
@show shadow_price(c2);

That was pretty easy and all the results can be queried with a solver independent API. Below let's go over what we just did, step by step.

## Package Importing
To write `JuMP.jl` programs, we'll need to import `JuMP` and an appropriate solver package to solve to the model. Hence, in this case we import `JuMP` and `HiGHS`: 

In [ ]:
using JuMP, HiGHS

## `JuMP.jl` Models
`JuMP` builds problems incrementally in a `Model` object. Create a model by passing an optimizer to the `Model` function:

In [ ]:
model = Model(HiGHS.Optimizer)

Here, the convention for the optimizer input is `SolverName.Optimizer`.

## Decision Variables
`JuMP.jl`'s modeling API principally uses macros to provide an intuitive symbolic interface. For adding/creating optimization variables, we use the `@variable` macro. To define, $x \geq 0$ we write:

In [ ]:
@variable(model, x >= 0)

To add $0 \leq y \leq 30$, we can write:

In [ ]:
@variable(model, 0 ≤ y ≤ 30)

Notice that we used `≤` (from `\leq` and pressing [TAB]) instead of `<=` to highlight how we can use unicode characters instead if we prefer.

## Objective
The objective function is specified via `@objective`. Hence, to set $\text{min} \; 12x + 20y$ we write:

In [ ]:
@objective(model, Min, 12x + 20y)

## Constraints
Constraints are added via `@constraint`. Here, we name our constraints `c1` and `c2` for convenience in querying results later on (this is optional and the argument can be omitted if wanted).

In [ ]:
@constraint(model, c1, 6x + 8y >= 100)

In [ ]:
@constraint(model, c2, 7x + 12y >= 120)

## Printing the Model
Simply showing the model results in a summary of what components it has:

In [ ]:
model

We have model with 2 optimization variables, an affine minimization objective, and 5 constraints of three different types. 

More conveniently we can print the model using `print`:

In [ ]:
print(model)

That is certainly more human-readable. Since, we are using a Jupyter notebook, we can even print the latex formulation of our model using `latex_formulation`:

In [ ]:
latex_formulation(model)

## Optimization
Now that we have a model, let's optimize it using `optimize!`:

In [ ]:
optimize!(model)

We will review methods later to specify solver options. One common one is `set_silent` which turns off the raw solver output:

In [ ]:
set_silent(model)
optimize!(model)

## Querying Results
Our model is now optimized, so let's see what happened using `JuMP.jl`'s general purpose query API. 

We can see the final status of the solver (i.e., why it stopped) using `termination_status`:

In [ ]:
termination_status(model)

Here, it stopped because it found the optimal solution. For a list of the possible statuses see https://jump.dev/JuMP.jl/stable/moi/reference/models/#MathOptInterface.TerminationStatusCode.

We can also check whether the solver found a primal feasible point using `primal_status`:

In [ ]:
primal_status(model)

It did find a feasible point. We can make the same check for the dual problem via `dual_status`:

In [ ]:
dual_status(model)

We also found a dual feasible point. The list of possible statuses is provided at https://jump.dev/JuMP.jl/stable/moi/reference/models/#MathOptInterface.ResultStatusCode.

To keep things simple, we can even just use `is_solved_and_optimal`:

In [ ]:
is_solved_and_feasible(model)

Now we know that we have an optimal solution with feasible primal and dual solutions that we can interrogate.

Query the objective value via `objective_value`:

In [ ]:
objective_value(model)

Now find the variable values using `value`:

In [ ]:
@show value(x)
@show value(y);

Finally, we can learn about the dual solution using `shadow_price`:

In [ ]:
@show value(c1)
@show value(c2);

We could have instead used `dual` to get the raw dual values, but `shadow_price` corrects the signs in accordance with the objective sense to have a consistent interpretation.

## Exercise: Simple QP Model
**Problem**
- Solve the following model using `JuMP.jl` using the `HiGHS` solver

$$
\begin{aligned}
& \min && 3x^2 + 2y^2 - 4x \\
& \;\;\text{s.t.} && 6x - 8y \geq 100 \\
&&& x + 12y = 120 \\
&&& x \geq 0 \\
&&& y \in [0, 3] \\
\end{aligned}
$$

In [ ]:
# PUT CODE HERE


## Working with Solvers
We often not only want to specify a solver, but also want to set some attributes as well. Here, the attributes are solver specific and can be found by checking the documentation associated with each solver. We can also specify/modify attributes using `set_attribute`:

In [ ]:
model = Model(HiGHS.Optimizer)
set_attribute(model, "output_flag", false)
set_attribute(model, "presolve", "on")

For convenience, `JuMP.jl` provides a few solver-agnostic methods for setting common attributes such as turning the output off and setting a time limit:

In [ ]:
model = Model(HiGHS.Optimizer)
set_silent(model) # turn the output printing off
set_time_limit_sec(model, 60.0) # set a time limit

What about the "good" Ipopt? We can change out the solve Ipopt uses in Julia using automatic compiler packages that automate the set up of different linear solvers: https://github.com/jump-dev/Ipopt.jl?tab=readme-ov-file#linear-solvers

## Variables
Let's take a deeper dive into more of the things we can do with `@variable`.

### Containers and Sets
We have already seen how to add individual scalar variables, now let's see how to add multiple variables at once.

`JuMP.jl` uses 3 data structures to store variable collections:
- `Array`s: The native Julia arrays
- `DenseAxisArray`s: Dense arrays with arbitrary indices
- `SparseAxisArray`s: Sparse arrays with arbitrary indices

Arrays are created using integer indices of the form `1:n`. For instance, the matrix:

In [ ]:
model = Model()
@variable(model, a[1:2, 1:4])

This creates a 2 x 4 matrix of variables that is stored to `a` which we can index and use in defining our problem.

We can also create an n-dimensional vector variable $x \in \mathbb{R}^n$ with upper and lower bounds:

In [ ]:
n = 5
l = [1, 2, 3, 4, 5]
u = [10, 11, 12, 13, 14]

@variable(model, l[i] <= x[i = 1:n] <= u[i])

Notice we declare an index `i` to help us define the appropriate values. 

We can use other index forms that don't conform to `1:n` and make `DenseAxisArray`s:

In [ ]:
@variable(model, z[i = 2:3, j = 1:2:3] >= i + 2j)

We don't even have to use integers:

In [ ]:
@variable(model, w[["red", "blue"], 1:5] <= 1)

For indices that do not form a rectangular set, a `SparseAxisArray` is formed:

In [ ]:
@variable(model, u[i = 1:2, j = i:3])

We can even add a conditional statement after a `;` when defining indices:

In [ ]:
@variable(model, v[i = 1:3, j = 1:4; i + j <= 4])

### Integrality
To specify integer variables, we need only add the `Int` argument:

In [ ]:
@variable(model, integer_x, Int)

Similarly, we create binary variables via the `Bin` argument:

In [ ]:
@variable(model, binary_x, Bin)

### Exercise: Nodal Variables
**Problem**
- Create a variable named `xp`
- `xp` should be integer valued between 0 and 3
- `xp` should be indexed over each arc `(i, j)` in `arcs`

In [ ]:
arcs = [(1, 2), (1, 3), (3, 2), (2, 4)]

# PUT CODE HERE


### Other Options
There are a variety of other things we can do with variables. We can create a fixed variable:

In [ ]:
@variable(model, x_fixed == 42)

We can specify the initial guess to pass on to the solver via `start`:

In [ ]:
@variable(model, q, start = 2)

### Modify Variables
There are a variety of ways to change variables after they are created. Some common methods include:
- `set_lower_bound`
- `set_upper_bound`
- `fix`
- `set_start_value`
- `set_binary`
- `set_integer`
- `delete`

For example:

In [ ]:
@variable(model, 🐦)
set_upper_bound(🐦, 10)
set_integer(🐦)
delete(model, 🐦)

There are many more things we can do, see https://jump.dev/JuMP.jl/stable/manual/variables/ to learn more. 

## Expressions
Sometimes we may want to use a mathematical expression in multiple constraints and/or the objective. We can create expressions using `@expression`. To motivate this, let's create a model with variables:

In [ ]:
model = Model()
@variable(model, x[1:2]);

We can create expressions using `@expression`. For instance:

In [ ]:
my_expr = @expression(model, x[1]^2 - 3x[2])

creates an anonymous expression that we can use elsewhere. We can also create named/registered expressions by adding a name argument:

In [ ]:
@expression(model, my_expr, x[1]^2 - 3x[2])
@show my_expr
@show model[:my_expr]; # all named variables/expressions/constraints are "registered" and be retrieved from the model

We can also create a container of expressions, just like we can for variables:

In [ ]:
@expression(model, expr[i = 1:2], 4x[i]^2)

We can also use Julia's native linear algebra to define expressions:

In [ ]:
@variable(model, y[1:3])
A = [1 2 4; 2 6 1]

# Two ways to do the same thing
@expression(model, sum(x[i] * A[i, j] * y[j] for j in 1:3, i in 1:2)); # equivalent sum-based expression
@expression(model, x' * A * y)


Nonlinear expressions are made the same way:

In [ ]:
@expression(model, nlexpr[i = 1:2], 2sin(x[i]))

The library of built-in univariate operators is derived those listed in `MOI.ListOfSupportedNonlinearOperators`.

In [ ]:
import Ipopt

MOI.get(Ipopt.Optimizer(), MOI.ListOfSupportedNonlinearOperators())

If we want to use some other nonlinear operator that is not natively supported, we can add our own! For instance, let's add the `logerfcx` from `SpecialFunctions.jl` using `@operator`:

In [ ]:
using SpecialFunctions

@operator(model, op_logerfcx, 1, logerfcx) # register a univariate operator `op_logerfcx` and use auto differientiation for gradients
@expression(model, [i = 1:2], op_logerfcx(x[i]))

For more information on nonlinear expressions see https://jump.dev/JuMP.jl/stable/manual/nonlinear/.

## Objectives
We have already seen how to set objectives using `@objective`. If we want to change an objective, we can just call it again:

In [ ]:
# Model to play with
model = Model()
@variable(model, x[1:2])
@variable(model, y[1:3])

# Set objective
@objective(model, Min, log(x[1]) + x[2]^2)
@show objective_function(model)

# Change the objective
@objective(model, Min, 4x[1] + 3x[2])
@show objective_function(model);

If all we want to do is change a linear coefficient, then we can use `set_objective_coefficient` instead:

In [ ]:
set_objective_coefficient(model, x[1], 2)
objective_function(model)

### Exercise: Linear Algebra Objective
**Problem**
- Create a quadratic objective
- The function is $x^T A y + b^T y + c^Tx$
- Maximize the objective

In [ ]:
A = [1 3 6; -9 2 1]
b = [3, -2, 0]
c = [2, 1]

# PUT CODE HERE

### Parameters
Declaring parameters can a useful way to way the values of constants in a model without having reconstruct the whole thing. This can be accomplished via the `Parameter` set:

In [ ]:
@variable(model, p[i = 1:2] in Parameter(i))

We can query and update the values via `parameter_value` and `set_parameter_value`:

In [ ]:
@show parameter_value.(p)

set_parameter_value(p[2], 3.0)

@show parameter_value.(p);

We can use these in any expression/objective/constraint:

In [ ]:
@objective(model, Max, p[1] * x[1])
@expression(model, my_nl_expr, p[1] * x[2]^2)

There are some caveats to this parameter API, but nearly all its limitations are addressed by `ParametricOptInterface`: https://github.com/jump-dev/ParametricOptInterface.jl.

## Constraints
Previously, we saw how to add simple scalar constraints. We will now take a deeper dive into using constraints in `JuMP.jl`.

Let's setup a model and variables:

In [ ]:
model = Model()
@variable(model, x[1:2])
J = 2:3 # define a set-like index object
@variable(model, y[J]);

Commonly we define constraints with names:

In [ ]:
@constraint(model, c1, x[1] + 2x[2] >= 42)
model[:c1]

But we can keep them without a name and a pointer variable:

In [ ]:
c1 = @constraint(model, x[1] + 2x[2] >= 42)

### Constraint Abstraction
Constraints in `JuMP` (which are stored in the `MOI` backend) are stored with the form `function` in `set`. Here `function` can be any scalar/vector-valued algebraic expression and `set` describes the constraint placed on the expression. For instance, let's consider the linear constraint $x_1 + 2x_2 \geq 42$:

In [ ]:
@constraint(model, c2, x[1] + 2x[2] >= 42)

Let's interrogate the `function` and the `set`:

In [ ]:
raw_constr = constraint_object(c1)
@show jump_function(raw_constr)
@show moi_set(raw_constr);

So, we have a linear expression $x_1 + 2x_2$ and a constraint set $\geq 42$ which constrains the expression to be greater than 42. If we were so inclined, we could directly express the constraint this way:

In [ ]:
@constraint(model, x[1] + 2x[2] in MOI.GreaterThan(42.0))

To learn more about all the sets that `MOI` supports see https://jump.dev/JuMP.jl/stable/moi/manual/constraints/#Constraints-by-function-set-pairs. We will keep the remainder of the discussion to the symbolic forms that `JuMP` provides which conveniently wrap around these underlying sets.

A key consequence of this modeling abstraction is that `JuMP` *normalizes* constraints, moving variables to the right-hand side and moving constants to the left-hand side. For instance:

In [ ]:
@constraint(model, 2x[1] + 1 <= 4x[1] + 4)

### Constraint Senses
Here we review the symbolic senses supported by `@constraint`. We illustrate these below:

In [ ]:
@constraint(model, 4 <= 2 * x[2] <= 5)            # `lb <= expr <= ub` interval     (can also use `≤`)
@constraint(model, sum(x) <= 1)                   # `<=`               less than    (can also use `≤`)
@constraint(model, x[1] + 2 * x[2] >= 2)          # `>=`               greater than (can also use `≥`)
@constraint(model, sum(j * y[j] for j in J) == 3) # `==`               equal to

We can also use the vectorized version of these operators by adding a `.` in front of the operator. This is often useful with linear algebra definitions:

In [ ]:
A = [1 2; 3 4]
b = [5, 6]

@constraint(model, A * x .== b)

### Using Sets/Containers
In similar manner to expressions and variables, we can create collections of constraints using `JuMP` containers. For instance, consider the constraint $x_i^2 + 4y_j \leq 0, \ i \in \{1, 2\}, j \in J$:

In [ ]:
@constraint(model, my_constr[i ∈ 1:2, j ∈ J], x[i]^2 + 4y[j] ≤ 0)

Note that the name `my_constr` is optional and instead of `in` we used `∈`. Here the supported index syntax is exactly the same as `@variable`, in fact all the `JuMP` macros use the same syntax. We can access individual constraints by indexing the container we generate:

In [ ]:
my_constr[2, 3]

### Exercise: Arc Constraints
**Problem**
- Define constraints of form $2x_i + y_j = 0, \ (i, j) \in A$

In [ ]:
A = [(1, 2), (2, 3), (2, 2)]

# PUT CODE HERE

Note that it is also possible to modify constraints such as:
- `set_normalized_rhs`
- `set_normalized_coefficient`
- `delete`

### Other Constraints
We will now highlight other constraint types that are natively supported by `JuMP`.

First, consider second-order cone constraints $||x||_2 \leq t$:

In [ ]:
model = Model()
@variable(model, t)
@variable(model, x[1:2])
@constraint(model, [t; x] in SecondOrderCone())

Next, rotated second order cone constraints $||x||_2^2 \leq 2t \cdot u$:

In [ ]:
model = Model()
@variable(model, t)
@variable(model, u)
@variable(model, x[1:2])
@constraint(model, [t; u; x] in RotatedSecondOrderCone())

Next, semi-continuous variables $y \in \{0\} \cup [l, u]$ and semi-integer variables $z \in \{0\} \cup [l, l + 1, \dots, u]$:

In [ ]:
@variable(model, y)
@constraint(model, y in MOI.Semicontinuous(1.5, 3.5))
@variable(model, z)
@constraint(model, z in MOI.Semiinteger(1.0, 3.0))

Next, special ordered sets of type 1 (SOS1) and SOS2 constraints:

In [ ]:
@variable(model, v[1:3])
@constraint(model, v in SOS1())
@constraint(model, v in SOS2())

Next, indicator constraints where a linear constraint is enforced when a binary variable is 1:

In [ ]:
@variable(model, a, Bin)
@constraint(model, a => {y + z <= 1})
@constraint(model, !a => {z >= 3}) # inverted logic

Next, positive-semi definite (PSD) constraints:

In [ ]:
@variable(model, X[1:2, 1:2])
@constraint(model, X >= 0, PSDCone()) # note it is preferred to define as `@variable(X[1:2, 1:2], PSD)`

Finally, we'll mention complementarity constraints $F(s) \perp s$ with $s \in [lb, ub]$:

In [ ]:
@variable(model, 0 <= s <= 1)
@constraint(model, 2s - 1 ⟂ s)

I will also note that constraint programming constraints are also supported: https://jump.dev/JuMP.jl/stable/tutorials/linear/constraint_programming/.

## Solutions
We have already reviewed the common query methods which include:
- `termination_status`
- `primal_status`
- `dual_status`
- `objective_value`
- `value`
- `shadow_price`

Here we will take closer look and review a few more of the available methods.

Let's first setup an optimized model that we can query:

In [ ]:
model = Model(HiGHS.Optimizer)
set_silent(model)
@variable(model, x >= 0)
@variable(model, y[[:a, :b]] <= 1)
@objective(model, Max, -12x - 20y[:a])
@expression(model, my_expr, 6x + 8y[:a])
@constraint(model, my_expr >= 100)
@constraint(model, c1, 7x + 12y[:a] >= 120)
optimize!(model)

### Solution Summary
For a general overview, we can use `solution_summary`:

In [ ]:
solution_summary(model)

We can get even more information if we wish:

In [ ]:
solution_summary(model, verbose=true)

### Termination Status
We already discussed querying the statuses which are independent of the solver used. We can also extract the raw status as report by the solver via `raw_status`:

In [ ]:
raw_status(model)

### Primal/Dual Solutions
Before querying values, we should always check that there are some we can actually get via `has_values`:

In [ ]:
has_values(model)

To query the value of a container of a variable/expression/constraint collection, we broadcast over `value`:

In [ ]:
value.(y)

This returns a container with the same indices that contains the optimal values.

We can do the same thing with duals:

In [ ]:
@show has_duals(model)
@show dual(c1)

# Also check the duals of variable bound constraints
@show dual(LowerBoundRef(x))
@show dual.(UpperBoundRef.(y));

We should note that `JuMP`'s definition of dual depends on the constraint direction, not the objective sense (different from some linear programming conventions). If we want the other convention, we can use `shadow_price` and `reduced_cost` instead:

In [ ]:
@show shadow_price(c1)
@show reduced_cost(x)
@show reduced_cost.(y);

### Other Queries
Some other attributes we can query are:

In [ ]:
@show solve_time(model)
@show relative_gap(model)
@show simplex_iterations(model)
@show barrier_iterations(model)
@show node_count(model);

Some other things we can do which are beyond the scope of today include:
- Linear sensitivity analysis via `lp_sensitivity_report`
- Conflict analysis for infeasible models via `compute_conflict!`
- Feasibility checking via `primal_feasibility_report`
- For solver that return multiple solutions, we can use the `result` keyword to get the one we want

For more information see https://jump.dev/JuMP.jl/stable/manual/solutions/.

### Solver-Independent Callbacks
Callbacks can be powerful ways to modify the way optimization problems are solved. Typically, this is solver dependent, but `JuMP` provides a solver-independent API. In particular, three types of callbacks are supported:
- lazy constraints
- user-cuts
- heuristic solutions

Note that this is only supported with a few solvers such as CPLEX, GLPK, Gurobi, and Xpress. For details, see https://jump.dev/JuMP.jl/stable/manual/callbacks/.

## Extensions
To add to the capabilities of `JuMP`, there are a variety of extension packages:
- `MathOptAI.jl`: Embed ML models into JuMP
- `StochasticPrograms.jl`: Solve 2-stage stochastic programs
- `BilevelJuMP.jl`: Solve bi-level optimization problems
- `Coluna.jl`: Implement branch-and-price-and-cut approaches
- `Plasmo.jl`: Solve/decompose graph optimization models
- `PolyJuMP.jl`: Solve polynomial optimization problems
- `SDDP.jl`: Solve multi-stage stochastic problems via SDDP
- `SumOfSquares.jl`: Solve polynomial optimization problems
- `vOptGeneric.jl`: Multi-objective optimization
- `InfiniteOpt.jl`: Solve infinite-dimensional optimization problems
- `DisjunctiveProgramming.jl`: Solve GDP problems

We'll focus today on `InfiniteOpt.jl`!